# **Lab 4.2. Шаврин Алексей, группа 1306**

# Импорты

In [2]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_openml
from sklearn.model_selection import GridSearchCV, cross_val_score, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.cm as cm

np.random.seed(42)

# Загрузка исходных данных

In [ ]:
mnist = fetch_openml('mnist_784', version=1, as_frame=False, parser='auto', cache=True)
X = mnist.data
y = mnist.target.astype(np.uint8)

## Разделение на обучающую и тестовую выборки

In [5]:
X_train, X_test = X[:60000], X[60000:]
y_train, y_test = y[:60000], y[60000:]

shuffle_idx = np.random.permutation(len(X_train))
X_train = X_train[shuffle_idx]
y_train = y_train[shuffle_idx]

# Функция для сдвига данных

In [6]:
def shift_image(image, shift_x, shift_y):
    image = image.reshape(28, 28)
    shifted = np.zeros_like(image)
    if shift_y >= 0:
        dst_y_start, dst_y_end = shift_y, 28
        src_y_start, src_y_end = 0, 28 - shift_y
    else:
        dst_y_start, dst_y_end = 0, 28 + shift_y
        src_y_start, src_y_end = -shift_y, 28
    if shift_x >= 0:
        dst_x_start, dst_x_end = shift_x, 28
        src_x_start, src_x_end = 0, 28 - shift_x
    else:
        dst_x_start, dst_x_end = 0, 28 + shift_x
        src_x_start, src_x_end = -shift_x, 28
    shifted[dst_y_start:dst_y_end, dst_x_start:dst_x_end] = image[src_y_start:src_y_end, src_x_start:src_x_end]
    return shifted.reshape(784)

# Обертка над KNN с аугментацией

In [7]:
class AugmentedKNN(BaseEstimator, ClassifierMixin):
    def __init__(self, n_neighbors=5, weights='uniform', algorithm='auto'):
        self.n_neighbors = n_neighbors
        self.weights = weights
        self.algorithm = algorithm
        self.clf_ = None

    def fit(self, X, y):
        X_augmented = [X[i] for i in range(len(X))]
        y_augmented = [y[i] for i in range(len(y))]

        for i in range(len(X)):
            image = X[i]
            label = y[i]
            X_augmented.append(shift_image(image, 1, 0))
            y_augmented.append(label)
            X_augmented.append(shift_image(image, -1, 0))
            y_augmented.append(label)
            X_augmented.append(shift_image(image, 0, -1))
            y_augmented.append(label)
            X_augmented.append(shift_image(image, 0, 1))
            y_augmented.append(label)

        X_augmented = np.array(X_augmented, dtype=np.float64)
        y_augmented = np.array(y_augmented)

        self.clf_ = KNeighborsClassifier(
            n_neighbors=self.n_neighbors,
            weights=self.weights,
            algorithm=self.algorithm
        )
        self.clf_.fit(X_augmented, y_augmented)
        return self

    def predict(self, X):
        return self.clf_.predict(X)

    def score(self, X, y):
        return self.clf_.score(X, y)

# Масштабирование данных

In [8]:
X_train_scaled = X_train.astype(np.float64) / 255.0
X_test_scaled = X_test.astype(np.float64) / 255.0

# KNeighborsClassifier + GridSearchCV

In [ ]:
param_grid = {
    'n_neighbors': [1, 2, 3, 4, 5, 6],
    'weights': ['uniform', 'distance'],
    'algorithm': ['auto', 'brute']
}

base_knn = KNeighborsClassifier()

print("Выполняется поиск гиперпараметров с 3-кратной кросс-валидацией...")
grid_search = GridSearchCV(
    base_knn,
    param_grid,
    cv=3,
    scoring='accuracy',
    verbose=1,
    n_jobs=-1
)

grid_search.fit(X_train_scaled, y_train)

print(f"Лучшие параметры: {grid_search.best_params_}")
print(f"Лучшая точность на кросс-валидации: {grid_search.best_score_:.5f}")

# Оценка на тестовой выборке
best_knn = grid_search.best_estimator_
y_pred_knn = best_knn.predict(X_test_scaled)
test_accuracy = accuracy_score(y_test, y_pred_knn)

print(f"Точность на тестовой выборке: {test_accuracy:.5f}")

Выполняется поиск гиперпараметров с 3-кратной кросс-валидацией...
Fitting 3 folds for each of 24 candidates, totalling 72 fits
Лучшие параметры: {'algorithm': 'auto', 'n_neighbors': 4, 'weights': 'distance'}
Лучшая точность на кросс-валидации: 0.97192
Точность на тестовой выборке: 0.97140


# Аугментация данных

In [10]:
param_grid = {
    'n_neighbors': [1, 2, 3, 4, 5, 6],
    'weights': ['uniform', 'distance'],
    'algorithm': ['brute']
}

aug_knn = AugmentedKNN()

print("Поиск гиперпараметров на аугментированных данных...")
grid_search_aug = GridSearchCV(
    aug_knn,
    param_grid,
    cv=3,
    scoring='accuracy',
    verbose=1,
    n_jobs=-1
)

grid_search_aug.fit(X_train_scaled, y_train)

print(f"Лучшие параметры (с аугментацией): {grid_search_aug.best_params_}")
print(f"Лучшая точность на кросс-валидации (с аугментацией): {grid_search_aug.best_score_:.5f}")

# Оценка на тестовой выборке
best_knn_aug = grid_search_aug.best_estimator_
y_pred_aug = best_knn_aug.predict(X_test_scaled)
test_accuracy_aug = accuracy_score(y_test, y_pred_aug)

print(f"Точность на тестовой выборке (с аугментацией): {test_accuracy_aug:.5f}")

Поиск гиперпараметров на аугментированных данных...
Fitting 3 folds for each of 12 candidates, totalling 36 fits


/usr/local/lib/python3.12/dist-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


Лучшие параметры (с аугментацией): {'algorithm': 'brute', 'n_neighbors': 6, 'weights': 'distance'}
Лучшая точность на кросс-валидации (с аугментацией): 0.97908
Точность на тестовой выборке (с аугментацией): 0.97720
